In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder, TargetEncoder
from sklearn.compose import ColumnTransformer
from lightgbm import LGBMRegressor
from sklearn.pipeline import Pipeline
import pickle

In [2]:
df = pd.read_csv("data/final_house_data.csv")

In [3]:
def parse_age(age_str):
    if 'Sıfır' in age_str:
        return 0
    try:
        return int(age_str.split(' ')[0])
    except:
        return 0
df['building_age_num'] = df['building_age'].apply(parse_age)

In [4]:
def parse_rooms(room_str):
    try:
        parts = room_str.split('+')
        return sum([int(p.strip()) for p in parts])
    except:
        return 1
df['total_rooms'] = df['room_count'].apply(parse_rooms)

In [5]:
import re

def convert_floor_to_numeric(floor_str):
    if not isinstance(floor_str, str):
        return 0

    floor_str = floor_str.lower()

    if "bodrum" in floor_str:
        return -1
    elif "giriş" in floor_str or "zemin" in floor_str or "bahçe" in floor_str or "kot" in floor_str:
        return 1
    elif "ara kat" in floor_str:
        return 4
    elif "üst" in floor_str or "çatı" in floor_str:
        return 6
    elif "üzeri" in floor_str:
        return 21

    match = re.search(r"\d+", floor_str)
    if match:
        return int(match.group())
    return 1

df['floor_level_num'] = df['floor_level'].apply(convert_floor_to_numeric)

In [6]:
df["is_furnished"] = df["is_furnished"].fillna("Eşyalı Değil")

In [7]:
df["heating_type"] = df["heating_type"].replace("Merkezi (Pay Öl...", "Merkezi")

In [8]:
rare_classes = ['Doğalgaz Sobası', 'Klima', 'VRV', 'Kat Kaloriferi', 'Isıtma Yok', 'Soba', 'Belirtilmemiş']
df["heating_type"] = df["heating_type"].replace(rare_classes, "Diğer")

In [9]:
df_filtred = df[(df['price'] < 50000000) & (df['gross_sqm'] < 500)].copy()

In [10]:
features_to_drop = ["listing_id", "title", "room_count", "building_age", "city_name", "floor_level"]
df_model = df_filtred.drop(columns=features_to_drop, axis=1)

In [11]:
X = df_model.drop("price", axis=1)
y = df_model["price"]

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [13]:
target_cols = ["district_name", "neighborhood_name"]
ohe_cols = ["heating_type", "is_furnished"]

preprocessor = ColumnTransformer(
    transformers=[
        ("low_cardinality", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ohe_cols),
        ("high_cardinality", TargetEncoder(target_type="continuous", random_state=42), target_cols)
    ],
    remainder="passthrough"
)

In [14]:
lgb_model = LGBMRegressor(num_leaves=31, n_estimators=200, max_depth=-1, learning_rate=0.05, n_jobs=-1, verbose=-1, random_state=42)

pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', lgb_model)
    ])
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

C:\Users\VICTUS\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [15]:
r2_score(y_test, y_pred)

0.7820777913836187

In [16]:
with open("house_price_model.pkl","wb") as file:
    pickle.dump(pipeline, file)

In [17]:
df_model.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2019 entries, 0 to 2142
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   district_name      2019 non-null   object
 1   neighborhood_name  2019 non-null   object
 2   price              2019 non-null   int64 
 3   gross_sqm          2019 non-null   int64 
 4   net_sqm            2019 non-null   int64 
 5   heating_type       2019 non-null   object
 6   is_furnished       2019 non-null   object
 7   building_age_num   2019 non-null   int64 
 8   total_rooms        2019 non-null   int64 
 9   floor_level_num    2019 non-null   int64 
dtypes: int64(6), object(4)
memory usage: 173.5+ KB


In [18]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1615 entries, 812 to 1210
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   district_name      1615 non-null   object
 1   neighborhood_name  1615 non-null   object
 2   gross_sqm          1615 non-null   int64 
 3   net_sqm            1615 non-null   int64 
 4   heating_type       1615 non-null   object
 5   is_furnished       1615 non-null   object
 6   building_age_num   1615 non-null   int64 
 7   total_rooms        1615 non-null   int64 
 8   floor_level_num    1615 non-null   int64 
dtypes: int64(5), object(4)
memory usage: 126.2+ KB
